# Strategy Research Notebook

Use this notebook to:
- Explore market data
- Prototype new factor computations
- Run backtests and visualise results
- Conduct parameter sensitivity analysis

**Before running**: make sure you have installed all dependencies:
```bash
pip install -r ../requirements.txt
```

In [ ]:
import sys
sys.path.insert(0, '..')   # add project root to path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded OK')

## 1. Load configuration

In [ ]:
with open('../config/config.yaml') as f:
    config = yaml.safe_load(f)

print('Active strategy:', config['strategy']['active'])
print('Market mode:', config['market']['mode'])
print('Date range:', config['data']['start_date'], '→', config['data']['end_date'])

## 2. Fetch price data

In [ ]:
from src.data.us_fetcher import USFetcher
from src.data.preprocessor import Preprocessor

fetcher = USFetcher(
    cache_dir='../.cache/data',
    cache_enabled=True,
)

# Use a small universe for fast iteration
SYMBOLS = ['AAPL', 'MSFT', 'AMZN', 'NVDA', 'GOOGL', 'META', 'TSLA',
           'JPM', 'JNJ', 'V', 'PG', 'XOM', 'KO', 'HD', 'MCD',
           'BAC', 'WMT', 'CSCO', 'PEP', 'CVX']
BENCHMARK = 'SPY'

prices = fetcher.get_prices(
    SYMBOLS,
    start=config['data']['start_date'],
    end=config['data']['end_date'],
)
bench_prices = fetcher.get_prices([BENCHMARK],
    start=config['data']['start_date'],
    end=config['data']['end_date'],
)

prices = Preprocessor.clean(prices)
close = Preprocessor.get_close(prices)
print(f'Loaded {len(close.columns)} symbols, {len(close)} trading days')
close.tail(3)

## 3. Explore data

In [ ]:
# Normalised cumulative returns (base = 100)
cum_ret = (1 + close.pct_change()).cumprod() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

cum_ret.iloc[:, :5].plot(ax=axes[0], title='Top 5 stocks — cumulative return (base 100)')

# Correlation matrix
daily_returns = close.pct_change().dropna()
corr = daily_returns.corr()
im = axes[1].imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
axes[1].set_xticks(range(len(corr.columns)))
axes[1].set_yticks(range(len(corr.columns)))
axes[1].set_xticklabels(corr.columns, rotation=90, fontsize=7)
axes[1].set_yticklabels(corr.columns, fontsize=7)
axes[1].set_title('Return correlation matrix')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.show()

## 4. Factor exploration

In [ ]:
# Compute all built-in factors for the latest date
features = Preprocessor.compute_all_features(prices)
features.sort_values('momentum_12_1', ascending=False)

In [ ]:
# Factor correlation
fig, ax = plt.subplots(figsize=(8, 6))
corr_f = features.corr()
im = ax.imshow(corr_f.values, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_f.columns)))
ax.set_yticks(range(len(corr_f.columns)))
ax.set_xticklabels(corr_f.columns, rotation=30, ha='right')
ax.set_yticklabels(corr_f.columns)
ax.set_title('Factor cross-correlation')
plt.colorbar(im)
plt.tight_layout()
plt.show()

## 5. Run a backtest

In [ ]:
from src.backtest.engine import BacktestEngine

engine = BacktestEngine(config)
result = engine.run(
    prices=prices,
    strategy_name='momentum_12_1',
    benchmark_prices=bench_prices,
)

report = result.report()
print(f"Sharpe Ratio  : {report['sharpe_ratio']:.2f}")
print(f"Total Return  : {report['total_return']*100:.1f}%")
print(f"Max Drawdown  : {report['max_drawdown']*100:.1f}%")
if 'alpha' in report:
    print(f"Alpha (ann)   : {report['alpha']*100:.1f}%")
    print(f"Beta          : {report['beta']:.2f}")

In [ ]:
result.plot()

## 6. Compare strategies

In [ ]:
from src.strategies import list_strategies
print('Available strategies:', list_strategies())

In [ ]:
strategies_to_compare = ['momentum_12_1', 'mean_reversion', 'multi_factor']
summary_rows = []

for strategy_name in strategies_to_compare:
    try:
        r = engine.run(prices=prices, strategy_name=strategy_name, benchmark_prices=bench_prices)
        rep = r.report()
        summary_rows.append({
            'strategy': strategy_name,
            'total_return': f"{rep['total_return']*100:.1f}%",
            'ann_return': f"{rep['annualized_return']*100:.1f}%",
            'sharpe': f"{rep['sharpe_ratio']:.2f}",
            'max_dd': f"{rep['max_drawdown']*100:.1f}%",
            'alpha': f"{rep.get('alpha',0)*100:.1f}%",
        })
    except Exception as e:
        print(f'{strategy_name}: error — {e}')

pd.DataFrame(summary_rows).set_index('strategy')

## 7. Prototype a new strategy

Use this cell to test a factor computation before turning it into a full strategy file.

In [ ]:
# ===== PROTOTYPE AREA =====
#
# Example: 52-week high momentum (George & Hwang 2004)
# Buy stocks closest to their 52-week high.

close = Preprocessor.get_close(prices)

high_52w = close.iloc[-252:].max()
last_price = close.iloc[-1]
nearness = last_price / high_52w

print('52-week nearness scores (1.0 = at 52-week high):')
nearness.sort_values(ascending=False).head(10)

In [ ]:
# Validate: check IC (rank correlation with next-month returns)
from scipy.stats import spearmanr

fwd_ret = close.pct_change(21).shift(-21).iloc[-1]  # next-month return
common = nearness.index.intersection(fwd_ret.dropna().index)

ic, p = spearmanr(nearness[common], fwd_ret[common])
print(f'IC (Spearman rank correlation): {ic:.3f}  (p={p:.3f})')
print('Positive IC = factor predicts forward returns correctly')

## 8. Parameter sensitivity

Test how performance changes with different `top_n` values.

In [ ]:
import copy

sensitivity_results = []
for top_n in [5, 10, 20, 30]:
    cfg = copy.deepcopy(config)
    cfg['strategy']['momentum_12_1']['top_n'] = top_n
    eng = BacktestEngine(cfg)
    r = eng.run(prices=prices, strategy_name='momentum_12_1')
    rep = r.report()
    sensitivity_results.append({
        'top_n': top_n,
        'sharpe': rep['sharpe_ratio'],
        'ann_return': rep['annualized_return'],
        'max_dd': rep['max_drawdown'],
    })

df_sens = pd.DataFrame(sensitivity_results).set_index('top_n')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df_sens['sharpe'].plot(ax=axes[0], marker='o', title='Sharpe Ratio vs top_n')
(df_sens['ann_return'] * 100).plot(ax=axes[1], marker='o', title='Ann. Return (%) vs top_n')
(df_sens['max_dd'] * 100).plot(ax=axes[2], marker='o', title='Max Drawdown (%) vs top_n')
plt.tight_layout()
plt.show()
df_sens